In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\python\notebooks\yellow_tripdata_2015_01_clean.csv")
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

print(df.shape)
df.head()

# 05_Feature_Engineering

## Modules

1. Time Features
2. Business Time Features
3. Revenue Features
4. Trip Segmentation Features
5. Geographic Features
6. Final Validation
7. Export Feature Dataset

In [ ]:
# ==========================================
# Module 1 : Time Features
# ==========================================
# 1. Objective
# Create temporal features for demand analysis, peak-hour analysis,
# weekday/weekend comparison, forecasting and dashboard filtering.

# 2. Feature Creation
df['pickup_hour'] = df["tpep_pickup_datetime"].dt.hour
df['pickup_day'] = df["tpep_pickup_datetime"].dt.day
df['pickup_month'] = df["tpep_pickup_datetime"].dt.month
df['pickup_weekday'] = df["tpep_pickup_datetime"].dt.day_name()

df["weekend_flag"] = df["pickup_weekday"].isin(["Saturday" , "Sunday"])
print(df[
        [
            "pickup_hour",
            "pickup_day",
            "pickup_month",
            "pickup_weekday",
            "weekend_flag"
        ]
    ].head()
)
# 3. Feature Vaidation
df[[
    "pickup_hour",
    "pickup_weekday",
    "weekend_flag"
]].describe(include="all")
print("\nPickup Hour Distribution")
print(df['pickup_hour'].value_counts().sort_index())
print("\nPickup Weekday Distribution")
print(df['pickup_weekday'].value_counts())
print("\nWeekend Distribution")
print(df['weekend_flag'].value_counts())



## Key Observations

- All pickup hours are within the expected range (0–23).
- All seven weekdays are present.
- Weekend flag contains only True and False values.
- Feature validation indicates the derived columns were created successfully.

In [ ]:
# ==========================================
# Module 2 : Business Time Features
# ==========================================
# 1. Objective:
# Create business-oriented time features for congestion,
# demand analysis and operational reporting.
# Rush Hour Flag (7-10 AM and 4-7 PM)

# 2. Feature Creation
df['rush_hour_flag'] = (
    df['pickup_hour'].between(7,9)|
    df['pickup_hour'].between(16,18)
)
df['night_trip'] = (
    (df['pickup_hour']>=22)|
    (df['pickup_hour'] <5)
)
# 3. Feature Validation
print("Rush Hour Distribution")
print(df['rush_hour_flag'].value_counts())
print("\nNight Trip Distribution")
print(df['night_trip'].value_counts())
print("weekend Vs Weekday Trips")
print(df["weekend_flag"].value_counts())
print("\nWeekend Percentage")
print((df["weekend_flag"].value_counts(normalize=True) * 100).round(2))

## Key Observations
- Rush hour and night trip flags were successfully created.
- Both features contain only Boolean values (True/False).
- These features will support demand analysis, traffic analysis, and operational dashboards.

In [ ]:
# ==========================================
# Module 3 : Revenue Features
# ==========================================

# 1. Objective:
# Create revenue-related analytical features to evaluate
# pricing efficiency and customer tipping behaviour.
# 2. Feature Creation
import numpy as np
df['fare_per_mile'] = np.where(
    df['trip_distance'] >0 ,
    df["fare_amount"]/df['trip_distance'],
    np.nan
)
df["tip_percentage"] = np.where(
    df["fare_amount"] > 0,
    (df['tip_amount'] / df['fare_amount'])*100,
    np.nan
)
# 3. Feature Validation
print(df["tip_percentage"].describe())
print(df["fare_per_mile"].describe())


In [ ]:
# ==========================================
# Module 4 : Trip Segmentation
# Distance Distribution Analysis
# ==========================================

print(df['trip_distance'].describe(percentiles=[0.25 , 0.50 , 0.75 , 0.90 , 0.95 , 0.99]))
print("\n")
print(df['trip_duration_minutes'].describe(percentiles=[0.25 , 0.50 , 0.75 , 0.90 , 0.95 , 0.99]))


In [ ]:
print(df.dtypes)

In [8]:
df = df.rename(
    columns= {
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "RateCodeID": "rate_code_id"
    }
)
df.to_csv('yellow_tripdata_2015_01_featured.csv' , index = False)

In [7]:
rows_before = len(df)
df = df[ 
    (df["fare_amount"] <= 1000) &
    (df["tip_amount"] <= 500) &
    (df["total_amount"] <= 1500) &
    (df["tolls_amount"] <= 200)
]
rows_removed = rows_before - len(df)
print(f"Rows removed :{rows_removed:,}")

Rows removed :32


In [ ]:
print(df[
    ["fare_amount",
     "tip_amount",
     "total_amount",
     "trip_distance",
     "trip_duration_minutes"]
].describe( percentiles=[0.90, 0.95, 0.99, 0.999]))

In [9]:
print(len(df))

12739047


In [1]:
print(df['trip_distance'].mean())
print(df['trip_duration_minutes'].mean())
print(df['total_amount'].mean())
print(df['fare_amount'].mean())


NameError: name 'df' is not defined